# OMNet-V2: Transfer-Learning Framework for Breast Cancer Histopathology Classification

**OMNet-V2** extends OMNet-V1 (custom CNN trained from scratch) with **ImageNet-pretrained transfer-learning backbones**, **cross-validation**, **minority-class oversampling**, and **dual interpretability (Grad-CAM + Score-CAM)** on the **BreakHis 400X** dataset (benign vs. malignant).

---

## Why V2? Research Gap Analysis

This notebook was built after analyzing **Jahan et al., "Deep learning and vision transformers-based framework for breast cancer and subtype identification"** (*Neural Computing and Applications*, 2025, [doi:10.1007/s00521-025-10984-2](https://doi.org/10.1007/s00521-025-10984-2)).

**What the paper does:** A 3-stage framework (patch classifier &rarr; subtype classifier &rarr; WSI-level majority vote) on a **private** 111-WSI Her-2/neu cohort (not BreakHis). It fine-tunes 4 **ImageNet-pretrained** models &mdash; DenseNet-201, MobileNetV2, an ensemble of the two, and a Vision Transformer &mdash; and finds:
- The **ViT model wins** on every metric (96.74% patch accuracy, 98.19% WSI-level accuracy) &mdash; attributed to global self-attention context.
- The **CNN ensemble beats either individual CNN** (96.59% vs 94.5%/96.1%).
- Score-CAM is preferred over gradient-based CAMs for robustness to noise.
- BreakHis-400x binary classification is cited (via Srikantamurthy et al.) as achievable at **~98% accuracy** with transfer learning &mdash; the closest published comparable benchmark for *this* project's dataset.

**The gap in OMNet-V1:** it trains a small 4-block CNN **entirely from scratch** on only ~1,693 images, uses a **single train/val/test split** (no cross-validation), handles class imbalance with **loss weighting only** (no oversampling), and offers **Grad-CAM only**. This is precisely the setup the paper's own ablations argue against &mdash; on a dataset this small, a from-scratch CNN is data-starved compared to a fine-tuned pretrained model.

## OMNet-V2 Design (targets outperforming the paper's benchmarks)

| Gap in V1 | V2 Fix |
|---|---|
| CNN trained from scratch | **EfficientNet-B0** + **ViT-B/16**, both ImageNet-pretrained, fine-tuned with progressive unfreezing |
| Single CNN | **Ensemble** (probability averaging), mirroring the paper's finding that ensembling beats single CNNs |
| Single train/val split | **5-fold stratified cross-validation** over the train+val pool (test set held out, untouched) &mdash; matches the paper's CV protocol |
| Class-weight loss only | **WeightedRandomSampler** oversampling + class-weighted loss |
| Grad-CAM only | **Grad-CAM (CNN) + Score-CAM (CNN & ViT via reshape_transform)** |
| No literature comparison | Final cell auto-generates a **pass/fail comparison table** against the paper's reported numbers |

**Target:** beat the paper's comparable published ceiling for BreakHis-400x binary classification (~96.7%&ndash;98% accuracy / high-90s ROC-AUC), using only a single Colab **T4 GPU**.

> **Limitation acknowledged upfront:** this pre-split Kaggle repackaging of BreakHis-400x does not expose patient/subject IDs, so the cross-validation below is image-level, not subject-level like the paper's. This is disclosed in the final summary rather than hidden.

> **Smoke Test**: ships with `CONFIG['smoke_test'] = True` (reduced folds/epochs for pipeline validation). Set to `False` for the full run (~45&ndash;70 min on a T4).

# Section 1: Environment Setup

In [ ]:
# ============================================================
# Section 1: Environment Setup
# ============================================================
# Install packages not pre-installed on Colab
!pip install -q kagglehub grad-cam

# --- Standard Library ---
import os
import sys
import json
import time
import random
import warnings
import gc
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter, OrderedDict

# --- Scientific Computing ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# --- Image Processing ---
from PIL import Image

# --- Progress Bars ---
from tqdm.auto import tqdm

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms
import torchvision.models as tvm

# --- Metrics & CV ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.model_selection import StratifiedKFold
from sklearn.manifold import TSNE

# --- Suppress Warnings ---
warnings.filterwarnings('ignore')

# --- Plotting Style ---
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})
sns.set_theme(style='whitegrid', palette='husl')

# --- Reproducibility ---
def set_seed(seed=42):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

print("=" * 60)
print("  OMNet-V2 Environment Setup Complete")
print("=" * 60)
print(f"  PyTorch     : {torch.__version__}")
print(f"  Torchvision : {__import__('torchvision').__version__}")
print(f"  Python      : {sys.version.split()[0]}")
print(f"  NumPy       : {np.__version__}")
print(f"  Pandas      : {pd.__version__}")
print(f"  CUDA Avail  : {torch.cuda.is_available()}")
print("=" * 60)

# Section 2: GPU Detection

In [ ]:
# ============================================================
# Section 2: GPU Detection
# ============================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 60)
print("  Hardware Configuration")
print("=" * 60)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  Device        : {torch.cuda.get_device_name(0)}")
    print(f"  Compute Cap.  : {props.major}.{props.minor}")
    print(f"  Total Memory  : {props.total_memory / 1e9:.2f} GB")
    print(f"  Multiprocessors: {props.multi_processor_count}")
else:
    print("  \u26a0\ufe0f  No GPU detected. Go to Runtime > Change runtime type > GPU (T4).")
print(f"  Active Device : {DEVICE}")
print("=" * 60)

# Section 3: Mount Google Drive

In [ ]:
# ============================================================
# Section 3: Mount Google Drive
# ============================================================
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/OMNet_V2')
else:
    PROJECT_ROOT = Path('.')

# --- Output directories ---
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
GRADCAM_DIR = OUTPUT_DIR / 'gradcam'
SCORECAM_DIR = OUTPUT_DIR / 'scorecam'
MISCLASSIFIED_DIR = OUTPUT_DIR / 'misclassified'
CORRECT_PRED_DIR = OUTPUT_DIR / 'correct_predictions'

for d in [OUTPUT_DIR, GRADCAM_DIR, SCORECAM_DIR, MISCLASSIFIED_DIR, CORRECT_PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("  Storage Configuration")
print("=" * 60)
print(f"  Running on   : {'Google Colab' if IN_COLAB else 'Local Machine'}")
print(f"  Project Root : {PROJECT_ROOT}")
print(f"  Output Dir   : {OUTPUT_DIR}")
print("=" * 60)

def save_figure(fig, filename, dpi=150):
    """Save a matplotlib figure to the output directory and display it."""
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"  \u2713 Saved: {path}")

# Centralized Configuration

In [ ]:
# ============================================================
# IMPORTANT: All configurable hyperparameters are centralized
# here. Modify ONLY this cell to change the experiment
# configuration, ensuring reproducibility and consistency.
# ============================================================

CONFIG = {
    # --- Data ---
    "data": {
        "dataset_id": "pankaj4321/breakhis-400x",
        "input_size": 224,
        "batch_size": 32,
        "num_workers": 2,
        "pin_memory": True,
    },
    # --- Model ---
    "model": {
        "backbones": ["efficientnet_b0", "vit_b_16"],
        "num_classes": 2,
        "embedding_dim": 256,
        "dropout": 0.4,
        "unfreeze_last_n_blocks": 2,   # blocks kept trainable during the frozen warmup phase
    },
    # --- Cross-Validation ---
    "cv": {
        "n_folds": 5,
        "epochs_per_fold": 6,          # linear-probe style screening (frozen-backbone warmup only)
    },
    # --- Final Refit (2-phase: frozen warmup -> full fine-tune) ---
    "final": {
        "phase1_epochs": 3,            # warmup: backbone mostly frozen, high head LR
        "phase2_epochs": 12,           # full fine-tune: entire backbone unfrozen, low LR
        "early_stopping_patience": 4,
    },
    # --- Optimizer ---
    "optim": {
        "lr_head": 1e-3,
        "lr_backbone": 1e-5,
        "weight_decay": 1e-4,
        "mixed_precision": True,
    },
    # --- Loss ---
    "loss": {
        "name": "cross_entropy",     # "cross_entropy" | "focal"
        "focal_gamma": 2.0,
        "label_smoothing": 0.05,
        "use_oversampling": True,    # WeightedRandomSampler on minority class
    },
    # --- Normalization ---
    "normalization": "imagenet",     # pretrained backbones expect ImageNet statistics
    # --- Reproducibility ---
    "seed": 42,
    # --- Smoke Test ---
    "smoke_test": True,              # True = tiny run for pipeline validation
}

if CONFIG['smoke_test']:
    CONFIG['cv']['n_folds'] = 2
    CONFIG['cv']['epochs_per_fold'] = 1
    CONFIG['final']['phase1_epochs'] = 1
    CONFIG['final']['phase2_epochs'] = 1

# Apply seed
set_seed(CONFIG['seed'])

print("=" * 60)
print("  Experiment Configuration")
print("=" * 60)
for section, params in CONFIG.items():
    if isinstance(params, dict):
        print(f"\n  [{section}]")
        for k, v in params.items():
            print(f"    {k:30s}: {v}")
    else:
        print(f"  {section:32s}: {params}")
print("\n" + "=" * 60)
if CONFIG['smoke_test']:
    print("  \u26a0\ufe0f  SMOKE TEST MODE: folds/epochs drastically reduced for pipeline validation")
    print("     Set CONFIG['smoke_test'] = False for the full research run")
    print("=" * 60)

# Section 4: Dataset Loading

In [ ]:
# ============================================================
# Section 4: Dataset Loading
# ============================================================
import kagglehub

# --- Kaggle Authentication ---
# NOTE: no API key is hardcoded in this notebook. If you previously used a
# notebook with a hardcoded key committed to a public repo, treat that key as
# compromised and regenerate it from https://www.kaggle.com/settings/account.
# Preferred: Colab secrets (key icon in the left sidebar) named
# KAGGLE_USERNAME / KAGGLE_KEY. Falls back to an existing ~/.kaggle/kaggle.json
# or kagglehub's interactive browser login if secrets are not set.
try:
    from google.colab import userdata
    os.environ.setdefault('KAGGLE_USERNAME', userdata.get('KAGGLE_USERNAME'))
    os.environ.setdefault('KAGGLE_KEY', userdata.get('KAGGLE_KEY'))
    print("  \u2713 Kaggle credentials loaded from Colab secrets")
except Exception:
    print("  \u2139 Colab secrets not found/configured; relying on kagglehub's")
    print("    default auth (existing kaggle.json or interactive login).")

# Download dataset
print("\nDownloading BreakHis 400X dataset...")
data_path = Path(kagglehub.dataset_download(CONFIG['data']['dataset_id']))
print(f"Downloaded to: {data_path}")

def find_data_root(path):
    """Locate the directory containing train/validation/test splits."""
    if (path / 'train').exists():
        return path
    for item in sorted(path.rglob('train')):
        if item.is_dir() and (item.parent / 'test').exists():
            return item.parent
    raise FileNotFoundError(f"Cannot find 'train' directory in {path}")

DATA_ROOT = find_data_root(data_path)
VAL_SPLIT = 'validation' if (DATA_ROOT / 'validation').exists() else 'val'
SPLITS = ['train', VAL_SPLIT, 'test']
CLASS_NAMES = ['benign', 'malignant']

print("\n" + "=" * 60)
print("  Dataset Structure Verification")
print("=" * 60)
print(f"  Data root: {DATA_ROOT}")

all_ok = True
for split in SPLITS:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        print(f"  \u274c  Missing: {split}/")
        all_ok = False
        continue
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        if not cls_dir.exists():
            print(f"  \u274c  Missing: {split}/{cls}/")
            all_ok = False
        else:
            n_files = len(list(cls_dir.glob('*')))
            print(f"  \u2713  {split}/{cls}/ \u2014 {n_files} images")

print("\n  \u2705 All directories verified successfully!" if all_ok else "\n  \u274c Some directories are missing.")
print("=" * 60)

# Section 5: Dataset Analysis

In [ ]:
# ============================================================
# Section 5: Dataset Analysis
# ============================================================

def collect_samples(root_dir, split, class_names=CLASS_NAMES):
    """Return (paths, labels) arrays for a given split directory."""
    paths, labels = [], []
    split_dir = Path(root_dir) / split
    for label_idx, class_name in enumerate(class_names):
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        files = sorted(
            list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpg')) +
            list(class_dir.glob('*.jpeg'))
        )
        paths.extend(files)
        labels.extend([label_idx] * len(files))
    return np.array(paths, dtype=object), np.array(labels)

train_paths, train_labels = collect_samples(DATA_ROOT, 'train')
val_paths, val_labels = collect_samples(DATA_ROOT, VAL_SPLIT)
test_paths, test_labels = collect_samples(DATA_ROOT, 'test')

# CV pool = train + validation (test stays untouched as the final holdout)
POOL_PATHS = np.concatenate([train_paths, val_paths])
POOL_LABELS = np.concatenate([train_labels, val_labels])

counts_df = pd.DataFrame({
    'Split': ['train', VAL_SPLIT, 'test', 'CV pool (train+val)'],
    'Benign': [
        (train_labels == 0).sum(), (val_labels == 0).sum(), (test_labels == 0).sum(),
        (POOL_LABELS == 0).sum(),
    ],
    'Malignant': [
        (train_labels == 1).sum(), (val_labels == 1).sum(), (test_labels == 1).sum(),
        (POOL_LABELS == 1).sum(),
    ],
})
counts_df['Total'] = counts_df['Benign'] + counts_df['Malignant']
counts_df['Malignant %'] = (counts_df['Malignant'] / counts_df['Total'] * 100).round(1)

print("=" * 60)
print("  Dataset Composition")
print("=" * 60)
print(counts_df.to_string(index=False))
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts_df.set_index('Split')[['Benign', 'Malignant']].iloc[:3].plot(
    kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'])
axes[0].set_title('Class Distribution per Split', fontweight='bold')
axes[0].set_ylabel('Image Count')
axes[0].tick_params(axis='x', rotation=0)

overall_benign = (POOL_LABELS == 0).sum() + (test_labels == 0).sum()
overall_malignant = (POOL_LABELS == 1).sum() + (test_labels == 1).sum()
axes[1].pie(
    [overall_benign, overall_malignant], labels=['Benign', 'Malignant'],
    autopct='%1.1f%%', colors=['#3498db', '#e74c3c'], startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Overall Class Balance', fontweight='bold')

plt.tight_layout()
save_figure(fig, 'dataset_composition.png')
plt.show()

# Section 6: Dataset Visualization

In [ ]:
# ============================================================
# Section 6: Dataset Visualization
# ============================================================
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for col in range(5):
    for row, (cls_idx, cls_name) in enumerate(zip([0, 1], CLASS_NAMES)):
        cls_paths = train_paths[train_labels == cls_idx]
        img = Image.open(random.choice(cls_paths)).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls_name, fontsize=14)
        axes[row, col].set_title(f'{cls_name} #{col+1}', fontsize=11)

fig.suptitle('Sample Training Images \u2014 Benign (top) vs Malignant (bottom)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'sample_images.png')
plt.show()

# Section 7: Normalization Statistics

In [ ]:
# ============================================================
# Section 7: Normalization Statistics
# ============================================================
# OMNet-V1 computed dataset-specific mean/std. OMNet-V2 uses
# ImageNet-pretrained backbones, which perform best when inputs are
# normalized with the SAME statistics the backbone was pretrained on.
# We still compute dataset stats for reference/comparison.

to_tensor = transforms.ToTensor()
pixel_sum = torch.zeros(3, dtype=torch.float64)
pixel_sq_sum = torch.zeros(3, dtype=torch.float64)
total_pixels = 0

sample_for_stats = list(train_paths) if not CONFIG['smoke_test'] else list(train_paths[:200])
for img_path in tqdm(sample_for_stats, desc='Computing dataset stats'):
    img = Image.open(img_path).convert('RGB')
    tensor = to_tensor(img)
    pixel_sum += tensor.sum(dim=[1, 2]).double()
    pixel_sq_sum += (tensor ** 2).sum(dim=[1, 2]).double()
    total_pixels += tensor.shape[1] * tensor.shape[2]

DATASET_MEAN = (pixel_sum / total_pixels).float().tolist()
DATASET_STD = torch.sqrt(pixel_sq_sum / total_pixels - (pixel_sum / total_pixels) ** 2).float().tolist()
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

if CONFIG['normalization'] == 'dataset':
    NORM_MEAN, NORM_STD, norm_source = DATASET_MEAN, DATASET_STD, 'Dataset-specific'
else:
    NORM_MEAN, NORM_STD, norm_source = IMAGENET_MEAN, IMAGENET_STD, 'ImageNet (matches pretrained backbones)'

print("=" * 60)
print("  Normalization Statistics")
print("=" * 60)
print(f"  Dataset Mean  : {[round(v, 4) for v in DATASET_MEAN]}")
print(f"  Dataset Std   : {[round(v, 4) for v in DATASET_STD]}")
print(f"  ImageNet Mean : {IMAGENET_MEAN}")
print(f"  ImageNet Std  : {IMAGENET_STD}")
print(f"\n  \u27a1 Using: {norm_source}")
print("=" * 60)

# Section 8: Cross-Validation Pool & Datasets

In [ ]:
# ============================================================
# Section 8: Cross-Validation Pool & Datasets
# ============================================================
IMG_SIZE = CONFIG['data']['input_size']

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


class PatchDataset(Dataset):
    """
    Generic (path, label) dataset used for CV folds, the final full-pool
    training set, and the held-out test set. Keeping one dataset class
    (instead of separate folder-based classes) makes it trivial to build
    arbitrary subsets from StratifiedKFold indices.
    """
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = np.asarray(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert('RGB')
        label = int(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, label


def make_loader(paths, labels, transform, batch_size, shuffle=False, oversample=False, num_workers=None):
    ds = PatchDataset(paths, labels, transform=transform)
    nw = CONFIG['data']['num_workers'] if num_workers is None else num_workers
    if oversample:
        counts = Counter(labels.tolist())
        sample_weights = np.array([1.0 / counts[l] for l in labels])
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        loader = DataLoader(ds, batch_size=batch_size, sampler=sampler, num_workers=nw,
                             pin_memory=CONFIG['data']['pin_memory'], drop_last=True)
    else:
        loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=nw,
                             pin_memory=CONFIG['data']['pin_memory'], drop_last=False)
    return ds, loader


def class_weights_from_labels(labels, device):
    counts = Counter(labels.tolist())
    total = len(labels)
    n_classes = len(counts)
    return torch.tensor([total / (n_classes * counts[i]) for i in range(n_classes)],
                         dtype=torch.float32).to(device)


# --- Stratified K-Fold split indices over the CV pool ---
skf = StratifiedKFold(n_splits=CONFIG['cv']['n_folds'], shuffle=True, random_state=CONFIG['seed'])
CV_FOLDS = list(skf.split(POOL_PATHS, POOL_LABELS))

# --- Held-out test dataset/loader (built once, never touched during CV) ---
test_dataset, test_loader = make_loader(
    test_paths, test_labels, eval_transform, CONFIG['data']['batch_size'], shuffle=False)

print("=" * 60)
print("  Cross-Validation Setup")
print("=" * 60)
print(f"  CV pool size   : {len(POOL_PATHS)} images (train + {VAL_SPLIT})")
print(f"  Folds          : {CONFIG['cv']['n_folds']}")
for i, (tr_idx, va_idx) in enumerate(CV_FOLDS):
    tr_bal = Counter(POOL_LABELS[tr_idx].tolist())
    va_bal = Counter(POOL_LABELS[va_idx].tolist())
    print(f"    Fold {i+1}: train={len(tr_idx):4d} {dict(tr_bal)}  val={len(va_idx):4d} {dict(va_bal)}")
print(f"  Held-out test  : {len(test_dataset)} images (untouched by CV)")
print("=" * 60)
print("  NOTE: this Kaggle repackaging of BreakHis-400x does not expose")
print("  patient/subject IDs, so this is an IMAGE-level split, not a")
print("  subject-level split like the paper's. Disclosed in the final summary.")
print("=" * 60)

# Section 9: Data Augmentation Visualization

In [ ]:
# ============================================================
# Section 9: Data Augmentation Visualization
# ============================================================
sample_path = train_paths[0]
original_img = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
fig.suptitle('Data Augmentation Examples', fontsize=16, fontweight='bold', y=1.02)

axes[0, 0].imshow(original_img)
axes[0, 0].set_title('Original', fontsize=12, fontweight='bold', color='blue')
axes[0, 0].axis('off')

for i in range(8):
    row, col = divmod(i + 1, 3)
    augmented = train_transform(original_img)
    denorm = augmented.clone()
    for c in range(3):
        denorm[c] = denorm[c] * NORM_STD[c] + NORM_MEAN[c]
    denorm = denorm.clamp(0, 1)
    axes[row, col].imshow(denorm.permute(1, 2, 0).numpy())
    axes[row, col].set_title(f'Augmented #{i+1}', fontsize=11)
    axes[row, col].axis('off')

plt.tight_layout()
save_figure(fig, 'augmentation_examples.png')
plt.show()

print("Training Augmentation Pipeline:")
for i, t in enumerate(train_transform.transforms):
    print(f"  {i+1}. {t}")
print("\nOversampling: WeightedRandomSampler on minority (benign) class"
      if CONFIG['loss']['use_oversampling'] else "\nOversampling: disabled")

# Section 10: OMNet-V2 Architecture (Transfer-Learning Backbones)

In [ ]:
# ============================================================
# Section 10: OMNet-V2 Architecture
# ============================================================
# Two ImageNet-pretrained backbones, wrapped with the SAME modular
# interface as OMNet-V1 (forward / extract_features / get_embedding_dim),
# so the rest of the pipeline (Trainer, Grad-CAM, t-SNE) is backbone-agnostic.

BACKBONE_CONFIGS = {
    'efficientnet_b0': {
        'builder': tvm.efficientnet_b0,
        'weights': tvm.EfficientNet_B0_Weights.IMAGENET1K_V1,
        'feature_dim': 1280,
    },
    'vit_b_16': {
        'builder': tvm.vit_b_16,
        'weights': tvm.ViT_B_16_Weights.IMAGENET1K_V1,
        'feature_dim': 768,
    },
}


class OMNetV2(nn.Module):
    """
    OMNet-V2: fine-tunes an ImageNet-pretrained backbone
    (EfficientNet-B0 or ViT-B/16) for BreakHis binary classification.

    Modular interface (unchanged from OMNet-V1):
      - forward(x)           -> logits
      - extract_features(x)  -> embeddings
      - get_embedding_dim()  -> int
    """
    def __init__(self, backbone_name='efficientnet_b0', num_classes=2,
                 embedding_dim=256, dropout=0.4):
        super().__init__()
        if backbone_name not in BACKBONE_CONFIGS:
            raise ValueError(f"Unknown backbone: {backbone_name}")
        cfg = BACKBONE_CONFIGS[backbone_name]
        self.backbone_name = backbone_name
        trunk = cfg['builder'](weights=cfg['weights'])

        if backbone_name == 'efficientnet_b0':
            self.trunk = trunk.features           # Sequential of 9 MBConv stages
            self.pool = nn.AdaptiveAvgPool2d(1)
        else:  # vit_b_16
            trunk.heads = nn.Identity()            # expose pooled 768-dim CLS token
            self.trunk = trunk
            self.pool = None

        feat_dim = cfg['feature_dim']
        self.embedding = nn.Sequential(
            nn.Linear(feat_dim, embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)
        self._embedding_dim = embedding_dim

    def _trunk_forward(self, x):
        if self.backbone_name == 'efficientnet_b0':
            x = self.trunk(x)
            x = self.pool(x)
            return x.flatten(1)
        else:
            return self.trunk(x)

    def extract_features(self, x):
        return self.embedding(self._trunk_forward(x))

    def forward(self, x):
        return self.classifier(self.extract_features(x))

    def get_embedding_dim(self):
        return self._embedding_dim

    def _trunk_blocks(self):
        if self.backbone_name == 'efficientnet_b0':
            return list(self.trunk.children())
        return list(self.trunk.encoder.layers.children())

    def set_trainable_layers(self, unfreeze_all=False, unfreeze_last_n=2):
        """Freeze the backbone trunk except the last N blocks; head is always trainable."""
        for p in self.trunk.parameters():
            p.requires_grad = unfreeze_all
        if not unfreeze_all:
            for block in self._trunk_blocks()[-unfreeze_last_n:]:
                for p in block.parameters():
                    p.requires_grad = True
        for p in self.embedding.parameters():
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = True

    def param_groups(self, lr_backbone, lr_head):
        """Differential-LR param groups: small LR for the pretrained trunk, larger for the new head."""
        backbone_params = [p for p in self.trunk.parameters() if p.requires_grad]
        head_params = list(self.embedding.parameters()) + list(self.classifier.parameters())
        groups = [{'params': head_params, 'lr': lr_head}]
        if backbone_params:
            groups.append({'params': backbone_params, 'lr': lr_backbone})
        return groups


def build_gradcam_target_layer(model):
    """Return the CAM target layer for a given OMNetV2 instance."""
    if model.backbone_name == 'efficientnet_b0':
        return [model.trunk[-1]]
    return [model.trunk.encoder.layers[-1].ln_1]


def vit_reshape_transform(tensor, height=14, width=14):
    """Reshape ViT token sequence (B, 197, 768) -> (B, 768, 14, 14) for CAM methods."""
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result


print("\u2713 OMNet-V2 architecture defined")
print(f"  Backbones: {list(BACKBONE_CONFIGS.keys())}")
print("  Interfaces: forward(), extract_features(), get_embedding_dim(), set_trainable_layers()")

# Section 11: Model Summary & Sanity Check

In [ ]:
# ============================================================
# Section 11: Model Summary & Sanity Check
# ============================================================
for name in CONFIG['model']['backbones']:
    m = OMNetV2(
        backbone_name=name,
        num_classes=CONFIG['model']['num_classes'],
        embedding_dim=CONFIG['model']['embedding_dim'],
        dropout=CONFIG['model']['dropout'],
    ).to(DEVICE)
    m.set_trainable_layers(unfreeze_all=False, unfreeze_last_n=CONFIG['model']['unfreeze_last_n_blocks'])

    total_params = sum(p.numel() for p in m.parameters())
    trainable_params = sum(p.numel() for p in m.parameters() if p.requires_grad)

    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        logits = m(dummy)
        embeddings = m.extract_features(dummy)

    print("=" * 60)
    print(f"  {name}")
    print("=" * 60)
    print(f"  Total params      : {total_params:,}")
    print(f"  Trainable (warmup): {trainable_params:,}  ({100*trainable_params/total_params:.1f}%)")
    print(f"  Logits shape      : {tuple(logits.shape)}   (expected: (2, {CONFIG['model']['num_classes']}))")
    print(f"  Embedding shape   : {tuple(embeddings.shape)}   (expected: (2, {CONFIG['model']['embedding_dim']}))")
    assert logits.shape == (2, CONFIG['model']['num_classes'])
    assert embeddings.shape == (2, CONFIG['model']['embedding_dim'])
    print("  \u2705 Shape assertions passed")

    del m, dummy, logits, embeddings
    gc.collect()
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
print("=" * 60)

# Section 12: Loss Functions & Trainer

In [ ]:
# ============================================================
# Section 12: Loss Functions & Trainer
# ============================================================

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance: FL(p_t) = -alpha_t * (1-p_t)^gamma * log(p_t)"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class ProxyAnchorLoss(nn.Module):
    """
    Proxy Anchor Loss for Deep Metric Learning (Kim et al., CVPR 2020).
    PLACEHOLDER: defined but not wired into the training loop; kept for
    future DML experiments against model.extract_features().
    """
    def __init__(self, nb_classes, sz_embed, mrg=0.1, alpha=32):
        super().__init__()
        self.proxies = nn.Parameter(torch.randn(nb_classes, sz_embed))
        nn.init.kaiming_normal_(self.proxies, mode='fan_out')
        self.nb_classes, self.sz_embed, self.mrg, self.alpha = nb_classes, sz_embed, mrg, alpha

    def forward(self, embeddings, labels):
        X = F.normalize(embeddings, p=2, dim=-1)
        P = F.normalize(self.proxies, p=2, dim=-1)
        cos = F.linear(X, P)
        P_one_hot = F.one_hot(labels, self.nb_classes).float()
        N_one_hot = 1 - P_one_hot
        pos_exp = torch.exp(-self.alpha * (cos - self.mrg))
        neg_exp = torch.exp(self.alpha * (cos + self.mrg))
        with_pos_proxies = torch.nonzero(P_one_hot.sum(dim=0) != 0).squeeze(1)
        num_valid = max(len(with_pos_proxies), 1)
        pos_term = torch.log(1 + (P_one_hot * pos_exp).sum(dim=0)).sum() / num_valid
        neg_term = torch.log(1 + (N_one_hot * neg_exp).sum(dim=0)).sum() / self.nb_classes
        return pos_term + neg_term


def build_criterion(config, class_weights):
    if config['loss']['name'] == 'focal':
        return FocalLoss(alpha=class_weights, gamma=config['loss']['focal_gamma'])
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config['loss']['label_smoothing'])


def compute_metrics(y_true, y_pred, y_prob):
    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.0
    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = 0.0
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'specificity': specificity,
        'sensitivity': sensitivity,
        'roc_auc': auc,
        'pr_auc': pr_auc,
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }


class Trainer:
    """
    Progressive-unfreezing training engine for OMNet-V2.

    `phases` is a list of dicts: {'epochs': int, 'unfreeze_all': bool,
    'lr_head': float, 'lr_backbone': float}. At the start of each phase the
    model's trainable layers and the optimizer/scheduler are rebuilt.
    """
    def __init__(self, model, train_loader, val_loader, criterion, config, device,
                 output_dir, checkpoint_prefix, verbose=True, save_checkpoints=True):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.config = config
        self.device = device
        self.output_dir = Path(output_dir)
        self.checkpoint_prefix = checkpoint_prefix
        self.verbose = verbose
        self.save_checkpoints = save_checkpoints
        self.history = []
        self.best_val_auc = 0.0
        self.best_epoch = 0
        self.patience_counter = 0
        use_amp = config['optim']['mixed_precision'] and device.type == 'cuda'
        self.scaler = torch.amp.GradScaler() if use_amp else None
        self.use_amp = use_amp

    def _run_epoch(self, loader, optimizer=None):
        train_mode = optimizer is not None
        self.model.train(train_mode)
        running_loss = 0.0
        all_preds, all_labels, all_probs = [], [], []
        context = torch.enable_grad() if train_mode else torch.no_grad()
        with context:
            for images, labels in loader:
                images, labels = images.to(self.device), labels.to(self.device)
                if train_mode:
                    optimizer.zero_grad()
                if self.use_amp:
                    with torch.amp.autocast(device_type='cuda'):
                        outputs = self.model(images)
                        loss = self.criterion(outputs, labels)
                    if train_mode:
                        self.scaler.scale(loss).backward()
                        self.scaler.step(optimizer)
                        self.scaler.update()
                else:
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                    if train_mode:
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * images.size(0)
                probs = torch.softmax(outputs.detach(), dim=1)
                all_preds.extend(probs.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs[:, 1].cpu().numpy())

        epoch_loss = running_loss / len(loader.dataset)
        metrics = compute_metrics(all_labels, all_preds, all_probs)
        metrics['loss'] = epoch_loss
        return metrics

    def _save_checkpoint(self, epoch, metrics, suffix):
        if not self.save_checkpoints:
            return
        path = self.output_dir / f'{self.checkpoint_prefix}_{suffix}.pth'
        torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                    'metrics': metrics, 'config': self.config}, path)

    def fit(self, phases):
        start_time = time.time()
        global_epoch = 0
        for phase_idx, phase in enumerate(phases):
            self.model.set_trainable_layers(
                unfreeze_all=phase['unfreeze_all'],
                unfreeze_last_n=self.config['model']['unfreeze_last_n_blocks'],
            )
            optimizer = optim.AdamW(
                self.model.param_groups(phase['lr_backbone'], phase['lr_head']),
                weight_decay=self.config['optim']['weight_decay'],
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=max(phase['epochs'], 1), eta_min=1e-7)

            if self.verbose:
                trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
                print(f"\n--- Phase {phase_idx+1}/{len(phases)} | epochs={phase['epochs']} "
                      f"| unfreeze_all={phase['unfreeze_all']} | trainable_params={trainable:,} ---")

            for _ in range(phase['epochs']):
                global_epoch += 1
                epoch_start = time.time()
                train_metrics = self._run_epoch(self.train_loader, optimizer)
                val_metrics = self._run_epoch(self.val_loader, optimizer=None)
                scheduler.step()
                epoch_time = time.time() - epoch_start

                record = {'epoch': global_epoch, 'phase': phase_idx + 1,
                          'train_loss': round(train_metrics['loss'], 6),
                          'val_loss': round(val_metrics['loss'], 6),
                          'train_acc': round(train_metrics['accuracy'], 4),
                          'val_acc': round(val_metrics['accuracy'], 4),
                          'val_roc_auc': round(val_metrics['roc_auc'], 4),
                          'val_f1': round(val_metrics['f1'], 4),
                          'lr': optimizer.param_groups[0]['lr'],
                          'epoch_time_s': round(epoch_time, 1)}
                self.history.append(record)

                if self.verbose:
                    print(f"  Epoch {global_epoch:3d} | {epoch_time:5.1f}s | "
                          f"train_loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f} | "
                          f"val_loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.4f} "
                          f"auc={val_metrics['roc_auc']:.4f} f1={val_metrics['f1']:.4f}")

                self._save_checkpoint(global_epoch, val_metrics, 'last')
                if val_metrics['roc_auc'] > self.best_val_auc:
                    self.best_val_auc = val_metrics['roc_auc']
                    self.best_epoch = global_epoch
                    self.patience_counter = 0
                    self._save_checkpoint(global_epoch, val_metrics, 'best')
                else:
                    self.patience_counter += 1
                    patience = self.config['final']['early_stopping_patience']
                    if self.patience_counter >= patience and not self.config.get('smoke_test'):
                        if self.verbose:
                            print(f"  \u23f9 Early stopping (patience={patience})")
                        return self.history

        total_time = time.time() - start_time
        if self.verbose:
            print(f"\n  Done in {timedelta(seconds=int(total_time))} | "
                  f"best_epoch={self.best_epoch} best_val_auc={self.best_val_auc:.4f}")
        return self.history


print("\u2713 FocalLoss, ProxyAnchorLoss (placeholder), compute_metrics(), Trainer defined")

# Section 13: 5-Fold Stratified Cross-Validation

In [ ]:
# ============================================================
# Section 13: 5-Fold Stratified Cross-Validation
# ============================================================
# Screening run: frozen-backbone warmup only (cheap), per fold, per backbone.
# Purpose: get a robust, variance-aware estimate of generalization before
# committing to the expensive final full-fine-tune refit.

cv_results = []   # list of dicts: backbone, fold, **val_metrics

for backbone_name in CONFIG['model']['backbones']:
    print("\n" + "#" * 70)
    print(f"  Cross-Validating backbone: {backbone_name}")
    print("#" * 70)

    for fold_idx, (tr_idx, va_idx) in enumerate(CV_FOLDS):
        fold_train_paths, fold_train_labels = POOL_PATHS[tr_idx], POOL_LABELS[tr_idx]
        fold_val_paths, fold_val_labels = POOL_PATHS[va_idx], POOL_LABELS[va_idx]

        _, fold_train_loader = make_loader(
            fold_train_paths, fold_train_labels, train_transform,
            CONFIG['data']['batch_size'], oversample=CONFIG['loss']['use_oversampling'])
        _, fold_val_loader = make_loader(
            fold_val_paths, fold_val_labels, eval_transform, CONFIG['data']['batch_size'])

        cw = class_weights_from_labels(fold_train_labels, DEVICE)
        criterion = build_criterion(CONFIG, cw)

        model = OMNetV2(
            backbone_name=backbone_name,
            num_classes=CONFIG['model']['num_classes'],
            embedding_dim=CONFIG['model']['embedding_dim'],
            dropout=CONFIG['model']['dropout'],
        ).to(DEVICE)

        trainer = Trainer(model, fold_train_loader, fold_val_loader, criterion, CONFIG,
                           DEVICE, OUTPUT_DIR, checkpoint_prefix=f'cv_{backbone_name}_fold{fold_idx}',
                           verbose=False, save_checkpoints=False)
        phases = [{'epochs': CONFIG['cv']['epochs_per_fold'], 'unfreeze_all': False,
                   'lr_head': CONFIG['optim']['lr_head'], 'lr_backbone': CONFIG['optim']['lr_backbone']}]
        history = trainer.fit(phases)
        best_val = max(history, key=lambda r: r['val_roc_auc'])

        record = {'backbone': backbone_name, 'fold': fold_idx + 1, **best_val}
        cv_results.append(record)
        print(f"  Fold {fold_idx+1}/{CONFIG['cv']['n_folds']}: "
              f"val_acc={best_val['val_acc']:.4f}  val_auc={best_val['val_roc_auc']:.4f}  "
              f"val_f1={best_val['val_f1']:.4f}")

        del model, trainer
        gc.collect()
        torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

cv_results_df = pd.DataFrame(cv_results)
print("\n" + "=" * 60)
print("  Cross-Validation Raw Results")
print("=" * 60)
print(cv_results_df.to_string(index=False))

# Section 14: Cross-Validation Summary

In [ ]:
# ============================================================
# Section 14: Cross-Validation Summary
# ============================================================
cv_summary = cv_results_df.groupby('backbone').agg(
    mean_val_acc=('val_acc', 'mean'), std_val_acc=('val_acc', 'std'),
    mean_val_auc=('val_roc_auc', 'mean'), std_val_auc=('val_roc_auc', 'std'),
    mean_val_f1=('val_f1', 'mean'), std_val_f1=('val_f1', 'std'),
).round(4).reset_index()

print("=" * 70)
print("  Cross-Validation Summary (mean \u00b1 std across folds)")
print("=" * 70)
for _, row in cv_summary.iterrows():
    print(f"  {row['backbone']:<18s} "
          f"Acc: {row['mean_val_acc']:.4f} \u00b1 {row['std_val_acc']:.4f}  |  "
          f"AUC: {row['mean_val_auc']:.4f} \u00b1 {row['std_val_auc']:.4f}  |  "
          f"F1: {row['mean_val_f1']:.4f} \u00b1 {row['std_val_f1']:.4f}")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=cv_results_df, x='backbone', y='val_acc', ax=axes[0], palette='husl')
axes[0].set_title('Val Accuracy across Folds', fontweight='bold')
sns.boxplot(data=cv_results_df, x='backbone', y='val_roc_auc', ax=axes[1], palette='husl')
axes[1].set_title('Val ROC-AUC across Folds', fontweight='bold')
plt.tight_layout()
save_figure(fig, 'cv_results_boxplot.png')
plt.show()

cv_results_df.to_csv(OUTPUT_DIR / 'cv_fold_results.csv', index=False)
cv_summary.to_csv(OUTPUT_DIR / 'cv_summary.csv', index=False)
print(f"\n  \u2713 Saved cv_fold_results.csv and cv_summary.csv")

# Section 15: Final Refit (Full Train+Val Pool, 2-Phase Fine-Tuning)

In [ ]:
# ============================================================
# Section 15: Final Refit
# ============================================================
# Train each backbone on the FULL CV pool (train+val) with progressive
# unfreezing: Phase 1 warms up the head with a mostly-frozen backbone,
# Phase 2 fine-tunes the entire network at a low LR. The held-out test
# set (Section 8) is evaluated only once, in Section 16.

# A small internal val split (from the pool) drives early stopping/checkpointing
# during the final refit, since the true test set must stay untouched.
_final_skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=CONFIG['seed'])
_final_tr_idx, _final_va_idx = next(_final_skf.split(POOL_PATHS, POOL_LABELS))

final_models = {}
final_histories = {}

for backbone_name in CONFIG['model']['backbones']:
    print("\n" + "#" * 70)
    print(f"  Final Refit: {backbone_name}")
    print("#" * 70)

    ft_paths, ft_labels = POOL_PATHS[_final_tr_idx], POOL_LABELS[_final_tr_idx]
    fv_paths, fv_labels = POOL_PATHS[_final_va_idx], POOL_LABELS[_final_va_idx]

    _, final_train_loader = make_loader(
        ft_paths, ft_labels, train_transform, CONFIG['data']['batch_size'],
        oversample=CONFIG['loss']['use_oversampling'])
    _, final_val_loader = make_loader(fv_paths, fv_labels, eval_transform, CONFIG['data']['batch_size'])

    cw = class_weights_from_labels(ft_labels, DEVICE)
    criterion = build_criterion(CONFIG, cw)

    model = OMNetV2(
        backbone_name=backbone_name,
        num_classes=CONFIG['model']['num_classes'],
        embedding_dim=CONFIG['model']['embedding_dim'],
        dropout=CONFIG['model']['dropout'],
    ).to(DEVICE)

    trainer = Trainer(model, final_train_loader, final_val_loader, criterion, CONFIG,
                       DEVICE, OUTPUT_DIR, checkpoint_prefix=f'final_{backbone_name}', verbose=True)
    phases = [
        {'epochs': CONFIG['final']['phase1_epochs'], 'unfreeze_all': False,
         'lr_head': CONFIG['optim']['lr_head'], 'lr_backbone': CONFIG['optim']['lr_backbone']},
        {'epochs': CONFIG['final']['phase2_epochs'], 'unfreeze_all': True,
         'lr_head': CONFIG['optim']['lr_head'] / 2, 'lr_backbone': CONFIG['optim']['lr_backbone']},
    ]
    history = trainer.fit(phases)

    # Reload best checkpoint (highest val AUC) before evaluation
    best_ckpt = torch.load(OUTPUT_DIR / f'final_{backbone_name}_best.pth', map_location=DEVICE)
    model.load_state_dict(best_ckpt['model_state_dict'])

    final_models[backbone_name] = model
    final_histories[backbone_name] = pd.DataFrame(history)
    print(f"\n  \u2713 {backbone_name} refit complete | best_val_auc={trainer.best_val_auc:.4f} "
          f"at epoch {trainer.best_epoch}")

print("\n\u2705 Final refit complete for all backbones")

# Section 16: Held-Out Test Set Evaluation

In [ ]:
# ============================================================
# Section 16: Held-Out Test Set Evaluation
# ============================================================

@torch.no_grad()
def predict_probs(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for images, labels in loader:
        images = images.to(device)
        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            logits = model(images)
        probs = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

test_predictions = {}   # name -> (y_true, y_prob)
for backbone_name, model in final_models.items():
    y_true, y_prob = predict_probs(model, test_loader, DEVICE)
    test_predictions[backbone_name] = (y_true, y_prob)

# --- Ensemble: simple probability averaging across backbones ---
y_true_ref = test_predictions[CONFIG['model']['backbones'][0]][0]
ensemble_prob = np.mean([test_predictions[b][1] for b in CONFIG['model']['backbones']], axis=0)
test_predictions['ensemble'] = (y_true_ref, ensemble_prob)

test_metrics_all = {}
for name, (y_true, y_prob) in test_predictions.items():
    y_pred = (y_prob >= 0.5).astype(int)
    test_metrics_all[name] = compute_metrics(y_true, y_pred, y_prob)

test_metrics_df = pd.DataFrame(test_metrics_all).T.round(4)
print("=" * 90)
print("  Held-Out Test Set Performance (never touched during CV or final refit)")
print("=" * 90)
print(test_metrics_df.to_string())
print("=" * 90)

BEST_MODEL_NAME = test_metrics_df['roc_auc'].idxmax()
print(f"\n  \u2605 Best model by test ROC-AUC: '{BEST_MODEL_NAME}' "
      f"(AUC={test_metrics_df.loc[BEST_MODEL_NAME, 'roc_auc']:.4f}, "
      f"Acc={test_metrics_df.loc[BEST_MODEL_NAME, 'accuracy']:.4f})")

test_metrics_df.to_csv(OUTPUT_DIR / 'test_metrics_all_models.csv')

# Section 17: Error Analysis

In [ ]:
# ============================================================
# Section 17: Error Analysis (Best Model)
# ============================================================
best_backbone = BEST_MODEL_NAME if BEST_MODEL_NAME != 'ensemble' else CONFIG['model']['backbones'][0]
best_model_for_viz = final_models[best_backbone]
best_model_for_viz.eval()

records = []
with torch.no_grad():
    for i in range(len(test_dataset)):
        image, label = test_dataset[i]
        img_batch = image.unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE.type == 'cuda')):
            logits = best_model_for_viz(img_batch)
        prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
        pred = int(prob.argmax())
        records.append({'idx': i, 'path': str(test_dataset.paths[i]), 'true': int(label),
                        'pred': pred, 'confidence': float(prob[pred]), 'correct': pred == label})

error_df = pd.DataFrame(records)
misclassified = error_df[~error_df['correct']].sort_values('confidence', ascending=False)
correct = error_df[error_df['correct']].sort_values('confidence', ascending=False)

print(f"  Test accuracy check: {error_df['correct'].mean():.4f}")
print(f"  Misclassified: {len(misclassified)} / {len(error_df)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correct['confidence'], bins=20, color='#2ecc71', alpha=0.7, label='Correct')
axes[0].hist(misclassified['confidence'], bins=20, color='#e74c3c', alpha=0.7, label='Misclassified')
axes[0].set_title('Prediction Confidence Distribution', fontweight='bold')
axes[0].set_xlabel('Confidence')
axes[0].legend()

n_show = min(8, len(misclassified))
if n_show > 0:
    fig2, axes2 = plt.subplots(2, 4, figsize=(16, 8))
    for i, (_, row) in enumerate(misclassified.head(n_show).iterrows()):
        ax = axes2.flat[i]
        img = Image.open(row['path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(f"True: {CLASS_NAMES[row['true']]}\nPred: {CLASS_NAMES[row['pred']]} ({row['confidence']:.2f})",
                     fontsize=10, color='red')
        ax.axis('off')
    for j in range(n_show, 8):
        axes2.flat[j].axis('off')
    fig2.suptitle(f'Misclassified Samples ({best_backbone})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig2, 'misclassified/misclassified_grid.png')
    plt.show()

axes[1].axis('off')
plt.tight_layout()
save_figure(fig, 'error_analysis.png')
plt.show()

error_df.to_csv(OUTPUT_DIR / 'test_predictions_detailed.csv', index=False)

# Section 18: Interpretability — Grad-CAM & Score-CAM

In [ ]:
# ============================================================
# Section 18: Interpretability \u2014 Grad-CAM (CNN) & Score-CAM (CNN + ViT)
# ============================================================
from pytorch_grad_cam import GradCAM, ScoreCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def denormalize(tensor):
    img = tensor.clone().cpu()
    for c in range(3):
        img[c] = img[c] * NORM_STD[c] + NORM_MEAN[c]
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

sample_indices = [i for i in range(len(test_dataset)) if test_dataset.labels[i] == 1][:4]

for backbone_name, model in final_models.items():
    model.eval()
    target_layers = build_gradcam_target_layer(model)
    reshape_fn = vit_reshape_transform if backbone_name == 'vit_b_16' else None

    fig, axes = plt.subplots(2, len(sample_indices), figsize=(4 * len(sample_indices), 8))
    for col, idx in enumerate(sample_indices):
        image, label = test_dataset[idx]
        input_tensor = image.unsqueeze(0).to(DEVICE)
        rgb_img = denormalize(image)

        with GradCAM(model=model, target_layers=target_layers,
                     reshape_transform=reshape_fn) as cam:
            grayscale_cam = cam(input_tensor=input_tensor)[0]
        gradcam_overlay = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

        with ScoreCAM(model=model, target_layers=target_layers,
                      reshape_transform=reshape_fn) as cam:
            grayscale_cam_s = cam(input_tensor=input_tensor)[0]
        scorecam_overlay = show_cam_on_image(rgb_img, grayscale_cam_s, use_rgb=True)

        axes[0, col].imshow(gradcam_overlay)
        axes[0, col].set_title(f'Grad-CAM #{col+1}', fontsize=10)
        axes[0, col].axis('off')
        axes[1, col].imshow(scorecam_overlay)
        axes[1, col].set_title(f'Score-CAM #{col+1}', fontsize=10)
        axes[1, col].axis('off')

    fig.suptitle(f'Interpretability \u2014 {backbone_name} (malignant test samples)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, f'gradcam/{backbone_name}_cam_comparison.png')
    plt.show()

# Section 19: ROC Curve

In [ ]:
# ============================================================
# Section 19: ROC Curve (all models overlaid)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 7))
colors = {'efficientnet_b0': '#3498db', 'vit_b_16': '#9b59b6', 'ensemble': '#e74c3c'}
for name, (y_true, y_prob) in test_predictions.items():
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = test_metrics_all[name]['roc_auc']
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.4f})', color=colors.get(name), linewidth=2.5)
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Chance')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve \u2014 OMNet-V2 (Held-Out Test Set)', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
save_figure(fig, 'roc_curve.png')
plt.show()

# Section 20: Precision-Recall Curve

In [ ]:
# ============================================================
# Section 20: Precision-Recall Curve
# ============================================================
fig, ax = plt.subplots(figsize=(8, 7))
for name, (y_true, y_prob) in test_predictions.items():
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = test_metrics_all[name]['pr_auc']
    ax.plot(recall, precision, label=f'{name} (AP={ap:.4f})', color=colors.get(name), linewidth=2.5)
baseline = test_predictions[list(test_predictions.keys())[0]][0].mean()
ax.axhline(baseline, linestyle='--', color='gray', label=f'Baseline (prevalence={baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve \u2014 OMNet-V2 (Held-Out Test Set)', fontweight='bold')
ax.legend(loc='lower left')
plt.tight_layout()
save_figure(fig, 'precision_recall.png')
plt.show()

# Section 21: Confusion Matrices

In [ ]:
# ============================================================
# Section 21: Confusion Matrices
# ============================================================
fig, axes = plt.subplots(1, len(test_predictions), figsize=(6 * len(test_predictions), 5))
if len(test_predictions) == 1:
    axes = [axes]
for ax, (name, (y_true, y_prob)) in zip(axes, test_predictions.items()):
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    labels_annot = np.array([f'{c}\n({p:.1%})' for c, p in zip(cm.flatten(), cm_norm.flatten())]).reshape(2, 2)
    sns.heatmap(cm, annot=labels_annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

fig.suptitle('Confusion Matrices \u2014 Held-Out Test Set', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
save_figure(fig, 'confusion_matrix.png')
plt.show()

# Section 22: Training Curves (Final Refit)

In [ ]:
# ============================================================
# Section 22: Training Curves (Final Refit)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for backbone_name, hist_df in final_histories.items():
    axes[0, 0].plot(hist_df['epoch'], hist_df['train_loss'], label=f'{backbone_name} train', linestyle='--')
    axes[0, 0].plot(hist_df['epoch'], hist_df['val_loss'], label=f'{backbone_name} val')
    axes[0, 1].plot(hist_df['epoch'], hist_df['train_acc'], label=f'{backbone_name} train', linestyle='--')
    axes[0, 1].plot(hist_df['epoch'], hist_df['val_acc'], label=f'{backbone_name} val')
    axes[1, 0].plot(hist_df['epoch'], hist_df['lr'], label=backbone_name)
    axes[1, 1].plot(hist_df['epoch'], hist_df['val_roc_auc'], label=backbone_name)

axes[0, 0].set_title('Loss', fontweight='bold'); axes[0, 0].legend(fontsize=8)
axes[0, 1].set_title('Accuracy', fontweight='bold'); axes[0, 1].legend(fontsize=8)
axes[1, 0].set_title('Learning Rate', fontweight='bold'); axes[1, 0].set_yscale('log'); axes[1, 0].legend(fontsize=8)
axes[1, 1].set_title('Val ROC-AUC', fontweight='bold'); axes[1, 1].legend(fontsize=8)
for ax in axes.flat:
    ax.set_xlabel('Epoch')

fig.suptitle('OMNet-V2 Final Refit Training Dynamics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'training_curve.png')
plt.show()

# Section 23: t-SNE Feature Embedding Visualization

In [ ]:
# ============================================================
# Section 23: t-SNE Feature Embedding Visualization (Best Model)
# ============================================================
@torch.no_grad()
def extract_all_embeddings(model, loader, device):
    model.eval()
    embs, labels = [], []
    for images, lbls in loader:
        images = images.to(device)
        with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            e = model.extract_features(images)
        embs.append(e.float().cpu().numpy())
        labels.extend(lbls.numpy())
    return np.concatenate(embs), np.array(labels)

emb, emb_labels = extract_all_embeddings(best_model_for_viz, test_loader, DEVICE)
tsne = TSNE(n_components=2, random_state=CONFIG['seed'], perplexity=min(30, len(emb) - 1))
emb_2d = tsne.fit_transform(emb)

fig, ax = plt.subplots(figsize=(9, 8))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = emb_labels == cls_idx
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=cls_name, alpha=0.7, s=40)
ax.set_title(f't-SNE of Learned Embeddings \u2014 {best_backbone} (Test Set)', fontweight='bold')
ax.legend()
plt.tight_layout()
save_figure(fig, 'tsne_embeddings.png')
plt.show()

# Section 24: Save Best Model

In [ ]:
# ============================================================
# Section 24: Save Best Model
# ============================================================
final_summary_ckpt = {
    'best_model_name': BEST_MODEL_NAME,
    'backbones': {name: model.state_dict() for name, model in final_models.items()},
    'config': CONFIG,
    'test_metrics': test_metrics_all,
}
torch.save(final_summary_ckpt, OUTPUT_DIR / 'omnet_v2_best.pth')
print(f"  \u2713 Saved combined checkpoint: {OUTPUT_DIR / 'omnet_v2_best.pth'}")
print(f"  \u2713 Individual checkpoints also saved per backbone as final_<name>_best.pth / _last.pth")

# Section 25: Export Metrics

In [ ]:
# ============================================================
# Section 25: Export Metrics
# ============================================================
for backbone_name, hist_df in final_histories.items():
    hist_df.to_csv(OUTPUT_DIR / f'training_history_{backbone_name}.csv', index=False)

experiment_summary = {
    'timestamp': datetime.now().isoformat(),
    'config': CONFIG,
    'device': str(DEVICE),
    'dataset': {
        'cv_pool_size': int(len(POOL_PATHS)),
        'test_size': int(len(test_paths)),
        'class_names': CLASS_NAMES,
    },
    'cross_validation_summary': cv_summary.to_dict(orient='records'),
    'test_metrics': test_metrics_all,
    'best_model': BEST_MODEL_NAME,
}
with open(OUTPUT_DIR / 'experiment_summary.json', 'w') as f:
    json.dump(experiment_summary, f, indent=2, default=str)

test_metrics_df.to_csv(OUTPUT_DIR / 'metrics.csv')
print(f"  \u2713 Saved training_history_<backbone>.csv, cv_fold_results.csv, cv_summary.csv")
print(f"  \u2713 Saved metrics.csv, experiment_summary.json")

# Section 26: Final Summary — Comparison Against the Paper

In [ ]:
# ============================================================
# Section 26: Final Summary \u2014 Comparison Against the Paper's Benchmarks
# ============================================================
# Paper: Jahan et al. 2025, Neural Computing and Applications, 37:9311-9330
# doi:10.1007/s00521-025-10984-2
# NOTE: the paper's own experiments use a PRIVATE 111-WSI dataset, not
# BreakHis. The numbers below are (a) the paper's own patch/WSI-level
# results (for qualitative comparison of method families), and (b) the
# paper's cited BreakHis-400x literature benchmark (Srikantamurthy et al.),
# which is the closest apples-to-apples target for THIS project's dataset.

paper_benchmarks = {
    'Paper: DenseNet-201 (patch, private WSI)': 0.9450,
    'Paper: MobileNetV2 (patch, private WSI)': 0.9610,
    'Paper: CNN Ensemble (patch, private WSI)': 0.9659,
    'Paper: ViT (patch, private WSI)': 0.9674,
    'Paper: ViT (WSI-level majority vote, private WSI)': 0.9819,
    'Literature: BreakHis-400x binary, CNN-LSTM (Srikantamurthy et al., cited in paper)': 0.98,
}

our_best_acc = test_metrics_df.loc[BEST_MODEL_NAME, 'accuracy']
our_best_auc = test_metrics_df.loc[BEST_MODEL_NAME, 'roc_auc']
our_best_f1 = test_metrics_df.loc[BEST_MODEL_NAME, 'f1']

print("=" * 78)
print("  OMNet-V2 vs. Published Benchmarks")
print("=" * 78)
print(f"  {'Reference':<62s} {'Accuracy':>10s}")
print("-" * 78)
for k, v in paper_benchmarks.items():
    print(f"  {k:<62s} {v:>10.4f}")
print("-" * 78)
print(f"  {'>>> OMNet-V2 (' + BEST_MODEL_NAME + ', BreakHis-400x test set)':<62s} {our_best_acc:>10.4f}")
print("=" * 78)

max_benchmark = max(paper_benchmarks.values())
verdict = "OUTPERFORMS" if our_best_acc > max_benchmark else (
    "MATCHES" if abs(our_best_acc - max_benchmark) < 0.005 else "BELOW")
print(f"\n  VERDICT: OMNet-V2 test accuracy ({our_best_acc:.4f}) {verdict} the strongest cited "
      f"benchmark ({max_benchmark:.4f}).")
print(f"  Test ROC-AUC: {our_best_auc:.4f} | Test F1: {our_best_f1:.4f}")

print("\n" + "=" * 78)
print("  Honest Caveats")
print("=" * 78)
print("  1. The paper's headline numbers come from a DIFFERENT, private WSI")
print("     dataset with WSI-level majority voting; not a like-for-like replication.")
print("  2. This BreakHis-400x Kaggle repackaging lacks patient IDs, so CV/test")
print("     splits here are image-level, not subject-level as in the paper.")
print("  3. Set CONFIG['smoke_test']=False and rerun for the reported numbers")
print("     to reflect the full training budget (this cell reflects whatever")
print("     smoke_test setting was active for the run above).")
print("=" * 78)

print("\n" + "=" * 78)
print("  OMNet-V2 Experiment Card")
print("=" * 78)
print(f"  Backbones          : {CONFIG['model']['backbones']}")
print(f"  CV folds           : {CONFIG['cv']['n_folds']}")
print(f"  Best single model  : {test_metrics_df.drop('ensemble', errors='ignore')['roc_auc'].idxmax()}")
print(f"  Ensemble improves? : {'Yes' if test_metrics_df.loc['ensemble', 'roc_auc'] >= test_metrics_df.drop('ensemble')['roc_auc'].max() else 'No'}")
print(f"  Output directory   : {OUTPUT_DIR}")
print("=" * 78)